# 📋 Capstone 9.5: Registry Workflow

**Goal**: Register the best model, manage versions, and set up champion/challenger aliases.

---

In [ ]:
import mlflow
import mlflow.sklearn
from mlflow import MlflowClient
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

mlflow.autolog(disable=True)
client = MlflowClient()

wine = load_wine()
X = pd.DataFrame(wine.data, columns=wine.feature_names)
y = wine.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

MODEL_NAME = "wine-quality-capstone"
print("✅ Setup complete!")

In [ ]:
# MLflow Database Configuration
# All modules share a centralized SQLite database at the project root
import os

_db_path = os.path.normpath(
    os.path.join(os.getcwd(), "..", "mlflow.db")
).replace(chr(92), "/")
mlflow.set_tracking_uri(f"sqlite:///{_db_path}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

## 1. Find the Best Runs

In [ ]:
# Find top runs from our capstone experiment
all_runs = mlflow.search_runs(
    experiment_names=["capstone-wine-quality"],
    order_by=["metrics.accuracy DESC"]
)

# Filter to runs with logged models and exclude nested/comparison runs
model_runs = all_runs[
    all_runs["metrics.accuracy"].notna() &
    all_runs.get("tags.mlflow.parentRunId", pd.Series([None]*len(all_runs))).isna() &
    (all_runs.get("tags.stage", "") != "comparison") &
    (all_runs.get("tags.stage", "") != "data_preparation")
].head(5)

print("🏆 Top 5 runs:")
for _, run in model_runs.iterrows():
    name = run.get("tags.mlflow.runName", "unnamed")
    acc = run.get("metrics.accuracy", 0)
    print(f"  {name:.<40} acc={acc:.4f}  id={run['run_id'][:8]}...")

champion_run_id = model_runs.iloc[0]["run_id"]
challenger_run_id = model_runs.iloc[1]["run_id"] if len(model_runs) >= 2 else None

print(f"\nChampion Run ID: {champion_run_id}")
if challenger_run_id:
    print(f"Challenger Run ID: {challenger_run_id}")

## 2. Register Champion Model

In [ ]:
# Register the champion
champion_result = mlflow.register_model(
    model_uri=f"runs:/{champion_run_id}/model",
    name=MODEL_NAME
)

champion_version = champion_result.version
print(f"✅ Champion registered!")
print(f"   Model: {MODEL_NAME}")
print(f"   Version: {champion_version}")

In [ ]:
# Set alias and metadata
client.set_registered_model_alias(MODEL_NAME, "champion", champion_version)

client.update_registered_model(
    name=MODEL_NAME,
    description="Wine quality classifier (3 classes) trained on sklearn Wine dataset. "
                "Capstone project model using RandomForest with Optuna HPO."
)

client.update_model_version(
    name=MODEL_NAME,
    version=champion_version,
    description=f"Champion model from capstone. Source run: {champion_run_id[:8]}..."
)

client.set_registered_model_tag(MODEL_NAME, "team", "ml-engineering")
client.set_registered_model_tag(MODEL_NAME, "task", "classification")
client.set_registered_model_tag(MODEL_NAME, "dataset", "wine-quality")

print(f"🏆 Version {champion_version} is the CHAMPION!")

## 3. Register Challenger

In [ ]:
if challenger_run_id:
    challenger_result = mlflow.register_model(
        model_uri=f"runs:/{challenger_run_id}/model",
        name=MODEL_NAME
    )
    challenger_version = challenger_result.version
    
    client.set_registered_model_alias(MODEL_NAME, "challenger", challenger_version)
    client.update_model_version(
        name=MODEL_NAME,
        version=challenger_version,
        description=f"Challenger model from capstone. Source run: {challenger_run_id[:8]}..."
    )
    
    print(f"⚔️  Version {challenger_version} is the CHALLENGER!")
else:
    print("No challenger — only one model found.")

## 4. View Registry State

In [ ]:
# Display registry status
model_info = client.get_registered_model(MODEL_NAME)

print(f"📋 Registered Model: {model_info.name}")
print(f"   Description: {model_info.description}")
print(f"   Tags: {model_info.tags}")
print(f"")

all_versions = client.search_model_versions(f"name='{MODEL_NAME}'")
for v in all_versions:
    aliases = ", ".join(v.aliases) if v.aliases else "-"
    print(f"   Version {v.version}: aliases=[{aliases}] | {v.description or '-'}")

## 5. Load and Test Champion

In [ ]:
# Load by alias — production pattern
champion_model = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}@champion")

y_pred = champion_model.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f"🏆 Champion model predictions:")
print(f"   Test accuracy: {acc:.4f}")
print(f"\n   Sample predictions:")
for i in range(5):
    match = "✅" if y_pred[i] == y_test.iloc[i] else "❌"
    print(f"   {match} predicted={wine.target_names[y_pred[i]]}, actual={wine.target_names[y_test.iloc[i]]}")

## ➡️ Next: `06_deploy_and_predict.ipynb` — Serve the champion model